# TAMU ECE Embedding Model Fine-Tuning

This notebook fine-tunes `all-MiniLM-L6-v2` on TAMU ECE domain data.

## Steps
1. Check GPU is available
2. Install dependencies
3. Upload your `training_pairs.jsonl`
4. Fine-tune the model
5. Download the fine-tuned model

> **Before running**: Make sure you set Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Step 1 — Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

if not torch.cuda.is_available():
    print('\n⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Step 2 — Install dependencies
!pip install -q sentence-transformers

In [ ]:
# Step 3 — Upload training_pairs.jsonl
from google.colab import files

print('Upload your training_pairs.jsonl file:')
uploaded = files.upload()

import json

filename = list(uploaded.keys())[0]
pairs = []
with open(filename) as f:
    for line in f:
        pairs.append(json.loads(line))

print(f'\nLoaded {len(pairs)} training pairs')
print('\nSample:')
print('  Query   :', pairs[0]['query'])
print('  Positive:', pairs[0]['positive'][:120], '...')

In [ ]:
# Step 4 — Fine-tune
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

# Config
BASE_MODEL  = 'all-MiniLM-L6-v2'
OUTPUT_DIR  = './tamu-ece-embedder'
EPOCHS      = 3
BATCH_SIZE  = 64

# Load base model
print(f'Loading base model: {BASE_MODEL}')
model = SentenceTransformer(BASE_MODEL)

# Build training examples
examples = [
    InputExample(texts=[p['query'], p['positive']])
    for p in pairs
    if p.get('query') and p.get('positive')
]
print(f'Training examples: {len(examples)}')

train_dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss = MultipleNegativesRankingLoss(model)
warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)

print(f'\nStarting fine-tuning...')
print(f'  Epochs      : {EPOCHS}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Warmup steps: {warmup_steps}')
print()

model.fit(
    train_objectives=[(train_dataloader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=OUTPUT_DIR,
    show_progress_bar=True,
)

print(f'\n✅ Fine-tuning complete. Model saved to {OUTPUT_DIR}')

In [ ]:
# Step 5 — Quick sanity check
from sentence_transformers import SentenceTransformer, util

finetuned = SentenceTransformer(OUTPUT_DIR)
base      = SentenceTransformer(BASE_MODEL)

test_queries = [
    'How do I apply for the ECE PhD program?',
    'Who are the machine learning faculty?',
    'What research labs does TAMU ECE have?',
]
test_doc = pairs[0]['positive'][:300]

print('Similarity comparison (fine-tuned vs base):\n')
for q in test_queries:
    q_ft   = finetuned.encode(q, normalize_embeddings=True)
    q_base = base.encode(q, normalize_embeddings=True)
    d_ft   = finetuned.encode(test_doc, normalize_embeddings=True)
    d_base = base.encode(test_doc, normalize_embeddings=True)
    sim_ft   = float(util.cos_sim(q_ft, d_ft))
    sim_base = float(util.cos_sim(q_base, d_base))
    print(f'  {q[:55]}')
    print(f'    Base fine-tuned : {sim_ft:.4f}  |  Original: {sim_base:.4f}')
    print()

In [ ]:
# Step 6 — Zip and download the model
import shutil
from google.colab import files

print('Zipping model...')
shutil.make_archive('tamu-ece-embedder', 'zip', '.', 'tamu-ece-embedder')
print('Downloading...')
files.download('tamu-ece-embedder.zip')
print('\n✅ Done! Unzip and set EMBEDDING_MODEL=./finetune/tamu-ece-embedder in your .env')